
# Pulmonary Artery Pressure Waveform Analysis for Heart Failure Assessment

Goal:
- Load PAP waveforms from the SCG-RHC dataset
- Extract hemodynamic and waveform morphology features
- Analyze relationships between waveform features and clinical measurements
by: Priyanka Patel and Samitha 

In [1]:
import wfdb
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.signal import find_peaks
from scipy.integrate import trapezoid

plt.style.use("default")

print("Libraries Loaded")

Libraries Loaded


## Features To Extract

Basic Features
- sPAP
- dPAP
- mPAP
- Pulse Pressure

Nakayama Features
- Fractional Pulse Pressure
- Coefficient of Variation
- Time to Half Area

Derivative Features
- Max dP/dt
- Min dP/dt
- Max d²P/dt²
- Min d²P/dt²

Shape Features
- Rise Time
- Area Under Curve

In [2]:
record_path = "TRM107-RHC1/TRM107-RHC1"

record = wfdb.rdrecord(record_path)

print(record.sig_name)

print("Sampling Rate:", record.fs)

FileNotFoundError: [Errno 2] No such file or directory: 'c:\\Users\\PATELPX107\\TRM107-RHC1\\TRM107-RHC1.hea'

In [ ]:

signal_names = record.sig_name

pap_idx = signal_names.index("RHC_pressure")

pap = record.p_signal[:, pap_idx]

fs = record.fs

print("PAP Length:", len(pap))


In [ ]:

plt.figure(figsize=(15,5))

plt.plot(pap)

plt.title("Pulmonary Artery Pressure Waveform")
plt.xlabel("Sample")
plt.ylabel("Pressure (mmHg)")

plt.show()


In [ ]:

start = 10000
end = 14000

plt.figure(figsize=(15,5))

plt.plot(pap[start:end])

plt.title("Zoomed PAP Waveform")

plt.xlabel("Sample")
plt.ylabel("Pressure (mmHg)")

plt.show()



In [ ]:

peaks, properties = find_peaks(
    pap,
    distance=int(fs * 0.5)
)

print("Peaks Found:", len(peaks))



In [ ]:

segment = pap[:5000]

plt.figure(figsize=(15,5))

plt.plot(segment)

peak_subset = peaks[peaks < 5000]

plt.scatter(
    peak_subset,
    segment[peak_subset],
    color="red"
)

plt.title("Detected Peaks")

plt.xlabel("Sample")

plt.ylabel("Pressure")

plt.show()

In [ ]:
def extract_features(beat, fs):

    beat = np.asarray(beat)

    sPAP = np.max(beat)

    dPAP = np.min(beat)

    pulse_pressure = sPAP - dPAP

    mPAP_formula = (sPAP + (2 * dPAP)) / 3

    mPAP_true = np.mean(beat)

    fractional_pulse_pressure = pulse_pressure / mPAP_true

    cv = np.std(beat) / mPAP_true

    auc = trapezoid(beat)

    dPdt = np.gradient(beat)

    d2Pdt2 = np.gradient(dPdt)

    peak_index = np.argmax(beat)

    rise_time_ms = (peak_index / fs) * 1000

    return {

        "sPAP": sPAP,

        "dPAP": dPAP,

        "pulse_pressure": pulse_pressure,

        "mPAP_formula": mPAP_formula,

        "mPAP_true": mPAP_true,

        "fractional_pulse_pressure":
            fractional_pulse_pressure,

        "cv": cv,

        "auc": auc,

        "max_dPdt":
            np.max(dPdt),

        "min_dPdt":
            np.min(dPdt),

        "mean_dPdt":
            np.mean(dPdt),

        "max_d2Pdt2":
            np.max(d2Pdt2),

        "min_d2Pdt2":
            np.min(d2Pdt2),

        "rise_time_ms":
            rise_time_ms
    }

In [ ]:
rows = []

for i in range(len(peaks) - 1):

    start = peaks[i]

    end = peaks[i + 1]

    beat = pap[start:end]

    if len(beat) < 20:
        continue

    features = extract_features(
        beat,
        fs
    )

    rows.append(features)

df = pd.DataFrame(rows)

print(df.shape)

df.head()

In [ ]:
df.describe()

In [ ]:
df.hist(
    figsize=(18,10),
    bins=20
)

plt.tight_layout()

plt.show()

In [ ]:
corr = df.corr(
    numeric_only=True
)

corr

In [ ]:
import seaborn as sns

plt.figure(figsize=(12,10))

sns.heatmap(
    corr,
    cmap="coolwarm",
    center=0
)

plt.title("Feature Correlation Matrix")

plt.show()

In [ ]:

patient_summary = df.mean(
    numeric_only=True
)

patient_summary


In [ ]:
patient_summary.to_csv(
    "patient_summary.csv"
)

print("Saved patient_summary.csv")